Clean duplicated values (by using link as uid)

In [13]:
import pandas as pd
from io import StringIO


file_path = 'public.csv'
# Load the data
df = pd.read_csv(file_path)
print(f"Original entries: {len(df)}")

# Remove duplicates based on the 'link' column
# keep='first' ensures we stay with the first instance we found
df_cleaned = df.drop_duplicates(subset='link', keep='first')
print(f"Duplicates removed: {len(df) - len(df_cleaned)}")

# Save to a new CSV
df_cleaned.to_csv('public.csv', index=False)

print(f"Cleanup complete. Remaining entries: {len(df_cleaned)}")

Original entries: 12895
Duplicates removed: 0
Cleanup complete. Remaining entries: 12895


In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Car Price Analyzer", layout="wide")

@st.cache_data
def load_data():
    try:
        df = pd.read_csv('car_listings.csv')
        # Clean the data: Ensure Price and Mileage are numbers
        # Removing "kr", "mil", and spaces
        df['price_num'] = df['price'].str.replace(r'\D', '', regex=True).astype(float)
        df['mileage_num'] = df['mileage'].str.replace(r'\D', '', regex=True).astype(float)
        # Ensure Year is a string for discrete coloring in the legend
        df['year'] = df['year'].astype(str)
        return df
    except Exception as e:
        st.error(f"Error loading data: {e}")
        return None

df = load_data()

if df is not None:
    st.title("📈 Car Market Price vs. Mileage")

    # 1. Find elements with the same title
    # We'll count them so the user knows which buttons are worth clicking
    title_counts = df['title'].value_counts()
    common_titles = title_counts[title_counts > 1].index.tolist()

    st.sidebar.header("Filter by Common Models")
    
    # 2. Display them as buttons in the sidebar
    selected_title = None
    for title in common_titles:
        if st.sidebar.button(f"{title} ({title_counts[title]} listings)"):
            selected_title = title

    # 3. Handle the Button Press / Graphing
    if selected_title:
        st.subheader(f"Analysis for: {selected_title}")
        
        # Filter data for the selected title
        filtered_df = df[df['title'] == selected_title]

        # 4. Create the Graph
        # x-axis: Mileage, y-axis: Price, Color: Year
        fig = px.scatter(
            filtered_df,
            x="mileage_num",
            y="price_num",
            color="year",
            size_max=15,
            title=f"Price Trend for {selected_title}",
            labels={"mileage_num": "Mileage (mil)", "price_num": "Price (SEK)", "year": "Model Year"},
            hover_data=["title", "fuel", "gearbox"] # Info shown when hovering over dots
        )

        # Update layout for better visibility
        fig.update_layout(xaxis_title="Mileage (mil)", yaxis_title="Price (kr)")
        
        st.plotly_chart(fig, use_container_width=True)
        
        # Show the raw data for these specific cars
        st.dataframe(filtered_df[['title', 'price', 'specs', 'location']])
    else:
        st.info("👈 Click a car title in the sidebar to visualize the price/mileage relationship.")

else:
    st.warning("Please ensure 'car_listings.csv' exists and has 'title', 'price', 'mileage', and 'year' columns.")

2026-02-01 09:10:32.844 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-01 09:10:32.845 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-01 09:10:33.077 
  command:

    streamlit run C:\Users\Etion\AppData\Roaming\Python\Python314\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-02-01 09:10:33.078 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-01 09:10:33.078 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-01 09:10:33.079 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-01 09:10:33.079 Thread 'MainThread': missing ScriptRunContext! This warning can b